# 04 · Skills, and paying only for what you read

Notebook 03 made the case that a filesystem is where an agent keeps its work.

This one makes a stranger case: it is also where an agent keeps its
**instructions**.

The reason is arithmetic. A system prompt is sent on every model call. If you
want an agent to know your ticket format, your review standards, your Python
conventions and your escalation policy, and you put all of that in the prompt,
you pay for all of it on every turn — including the turns where none of it is
relevant.

In [1]:
import pathlib, sys
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

repo root: /Users/aseem/Documents/hubbleflow/standalone-projects/agentic-crew


## What a skill is

A directory with a `SKILL.md` in it. YAML frontmatter, then a markdown body.

In [2]:
skill = ROOT / "skills/roles/engineering-manager/scoping-a-request/SKILL.md"
text = skill.read_text()

print("=" * 66)
print(text[:text.index("---", 4) + 3])          # the frontmatter
print("=" * 66)
print(f"...then {len(text.splitlines())} lines of body, which is the part")
print("the model does NOT see until it asks.")

---
name: scoping-a-request
description: How to turn a vague founder message into a named project and a ticket, and how to decide whether to answer directly or bring in the PM. Read on your first message.
allowed_tools: [read_ticket, write_ticket, name_project, spawn_agent, escalate_to_founder]
---
...then 63 lines of body, which is the part
the model does NOT see until it asks.


## Progressive disclosure

At `before_agent`, `SkillsMiddleware` reads **only the frontmatter** of every
skill it can see, and injects a short index into the system prompt: name,
description, path.

That is all the model gets. If it decides a skill applies, it calls
`read_file` on the path and gets the body.

So the standing cost of a skill is its description line. The body is paid for
only on the turns where it is used.

In [3]:
total = 0
print(f"{'skill':<32} {'description':<10} {'body'}")
print("-" * 62)
for path in sorted(ROOT.glob("skills/**/SKILL.md")):
    text = path.read_text()
    end = text.index("---", 4) + 3
    head, body = text[:end], text[end:]
    total += len(body)
    print(f"{path.parent.name:<32} {len(head):>6} ch {len(body):>7} ch")
print("-" * 62)
print(f"{'':<32} {'':>6}    {total:>7} ch of body, loaded only on demand")

skill                            description body
--------------------------------------------------------------
working-in-the-workspace            258 ch    1280 ch
writing-a-ticket                    216 ch    1280 ch
python-service-conventions          244 ch    1298 ch
reviewing-a-change                  271 ch    1261 ch
scoping-a-request                   299 ch    2002 ch
clarifying-a-request                287 ch    1911 ch
writing-a-test-plan                 233 ch    1158 ch
--------------------------------------------------------------
                                             10190 ch of body, loaded only on demand


## Which skills a role can see

The layering is in one function, and what it *omits* is the point.

In [4]:
src = (ROOT / "agents/shared/agent_loop.py").read_text()
start = src.index("def _skill_sources")
print(src[start:src.index("def _as_prompt")].rstrip())

def _skill_sources(role: str) -> list[str]:
    """Which skill directories this role can see, in override order.

    Base skills first, then the role's own, so a role skill of the same name
    wins. Only directories that exist are returned · a missing one makes the
    middleware log a warning on every boot, which trains people to ignore
    warnings.

    Note what is *not* here: another role's directory. A backend engineer
    cannot read the Engineering Manager's scoping playbook, because it was
    never in its list. That is scoping by construction rather than by asking
    the model nicely.
    """
    on_disk = Path(SKILLS_ROOT)
    relative = [Path("base"), Path("roles") / role.replace("_", "-")]

    # Existence is checked on disk; the paths handed to the middleware are the
    # ones the *backend* understands, under the /skills mount.
    sources = [f"{SKILLS_MOUNT}/{rel}" for rel in relative if (on_disk / rel).is_dir()]
    if not sources:
        log.warning("harness.no_sk

In [5]:
import agents.shared.agent_loop as loop

# The resolver checks the image path; point it at the checkout instead.
loop.SKILLS_ROOT = str(ROOT / "skills")

for role in ("engineering_manager", "backend_engineer", "qa_engineer"):
    print(f"{role:<22} {loop._skill_sources(role)}")

engineering_manager    ['/skills/base', '/skills/roles/engineering-manager']
backend_engineer       ['/skills/base', '/skills/roles/backend-engineer']
qa_engineer            ['/skills/base', '/skills/roles/qa-engineer']


A backend engineer's list contains `skills/base` and its own directory. It does
not contain `skills/roles/engineering-manager`.

So it cannot read the scoping playbook — not because it is told not to, but
because the path was never in its list. That is worth contrasting with the
usual approach of writing "do not attempt to scope projects" into a prompt and
hoping.

## Does the model actually do this?

Give the EM six skills and one ambiguous request, then count what it opened.

In [6]:
import os
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import AIMessage

assert os.environ.get("GOOGLE_API_KEY"), "set GOOGLE_API_KEY (see .env)"

agent = create_deep_agent(
    model=ChatGoogleGenerativeAI(model="gemini-3.5-flash"),
    system_prompt="You are an Engineering Manager. Consult your skills.",
    backend=FilesystemBackend(root_dir=str(ROOT), virtual_mode=False),
    skills=[str(ROOT / "skills/base"), str(ROOT / "skills/roles/engineering-manager")],
)

result = await agent.ainvoke({"messages": [{"role": "user", "content":
    "A founder says: 'Make onboarding better'. What is your first move?"}]})

opened = [tc["args"].get("file_path", "") for m in result["messages"]
          if isinstance(m, AIMessage) for tc in (m.tool_calls or [])
          if tc["name"] == "read_file"]

print(f"Skills offered:  {len(list(ROOT.glob('skills/base/*/SKILL.md')) + list(ROOT.glob('skills/roles/engineering-manager/*/SKILL.md')))}")
print(f"Skills opened:   {len(opened)}")
for p in opened:
    print("   ", p.replace(str(ROOT), "."))

Skills offered:  3
Skills opened:   1
    ./skills/roles/engineering-manager/scoping-a-request/SKILL.md


It opened the one that matched. The others cost their description line and
nothing else.

## Writing a good description

The description is the only thing the model sees before it chooses, so it
should answer *when would I need this*, not *what is in here*.

> Good: "How to turn a vague founder message into a named project and a ticket"
> Bad: "Project scoping documentation"

The second one is accurate and useless. The model cannot tell from it whether
the request in front of it qualifies.

## What you now know

1. A skill is a directory with `SKILL.md`: frontmatter the model always sees,
   a body it reads on demand.
2. The standing cost is the description line. That is what makes it affordable
   to give an agent a lot of specific knowledge.
3. Which skills a role can see is decided by which directories are in its
   source list — scoping by construction, not by instruction.
4. The description is a *when*, not a *what*.

Next: how one agent becomes several, and why in this repo a sub-agent is a pod.